In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import torch
import numpy as np
from openmm.app import *
from openmm import *
from openmm.unit import *
import matplotlib.pyplot as plt

def compute_energies(coords, topology_path, plot=True):
    # Load topology and force field (no solvent model)
    pdb = PDBFile(topology_path + ".pdb")
    forcefield = ForceField('amber14-all.xml')

    # Use coords[0, 0] for initial heavy-atom positions
    initial_positions = [Vec3(*xyz) for xyz in coords[0, 0].cpu().numpy()] * nanometer

    # Create modeller and add hydrogens
    modeller = Modeller(pdb.topology, initial_positions)
    modeller.addHydrogens(forcefield)

    # Build system without solvent
    system = forcefield.createSystem(modeller.topology,
                                     nonbondedMethod=NoCutoff,
                                     constraints=HBonds)

    # Set up simulation infrastructure
    integrator = VerletIntegrator(0.001)
    platform = Platform.getPlatformByName('Reference')  # Change to 'CUDA' if desired
    simulation = Simulation(modeller.topology, system, integrator, platform)

    # Get atom indices
    all_atoms = list(modeller.topology.atoms())
    heavy_atom_count = pdb.topology.getNumAtoms()
    hydrogen_indices = [i for i, atom in enumerate(all_atoms) if atom.element == element.hydrogen]
    heavy_indices = [i for i in range(len(all_atoms)) if i not in hydrogen_indices]

    # Sanity check on tensor shape
    num_paths, path_length, atoms_in_tensor, _ = coords.shape
    assert atoms_in_tensor == heavy_atom_count, (
        f"Expected {heavy_atom_count} atoms but got {atoms_in_tensor}"
    )

    # Store energies
    energies = torch.zeros((num_paths, path_length))

    for i in range(num_paths):
        for j in range(path_length):
            # Get heavy atom positions from tensor (unitless floats)
            heavy_coords = coords[i, j].cpu().numpy()
            current_positions = simulation.context.getState(getPositions=True).getPositions(asNumpy=True)

            # Replace heavy atom coordinates
            for k in range(heavy_atom_count):
                current_positions[k] = Vec3(*heavy_coords[k])
            current_positions *= nanometer
            simulation.context.setPositions(current_positions)

            # Temporarily zero heavy atom masses to freeze them
            for idx in heavy_indices:
                system.setParticleMass(idx, 0.0 * dalton)

            # Minimize energy (hydrogens only)
            simulation.minimizeEnergy()

            # Restore heavy atom masses to something valid (not needed for energy eval)
            for idx in heavy_indices:
                system.setParticleMass(idx, 12.0 * dalton)  # safe dummy mass

            # Get energy
            state = simulation.context.getState(getEnergy=True)
            energy = state.getPotentialEnergy().value_in_unit(kilojoules_per_mole)
            energies[i, j] = energy

    # Plot
    if plot:
        plt.figure(figsize=(10, 6))
        for i in range(num_paths):
            plt.plot(range(path_length), energies[i].numpy(), label=f'Path {i+1}')
        plt.xlabel('Time step')
        plt.ylabel('Potential Energy (kJ/mol)')
        plt.title('Energy Profiles (vacuum, hydrogens minimized)')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return energies


In [7]:
coords = torch.load("saved_models/tetrapeptides_all_atom/main_eval_output_om_interpolate_test_april9_model_intrinsic_larger_fulldataset_flowmatching/sample-om_interpolate_AVGR.pt")
coords = coords.reshape(4, 100, -1, 3) / 10
energies = compute_energies(coords, "/data/sanjeevr/4AA_sim/AVGR/AVGR")


> /home/sanjeevr/miniforge3/envs/om-diffusion/lib/python3.12/site-packages/openmm/app/modeller.py(977)addHydrogens()
    975                         newAtoms[parent] = newAtom
    976                         import pdb; pdb.set_trace()
--> 977                         newPositions.append(deepcopy(self.positions[parent.index]))
    978                         if parent in parents:
    979                             # Match expected hydrogens with existing ones and find which ones need to be added.

*** TypeError: object of type 'list_iterator' has no len()
*** TypeError: object of type 'list_iterator' has no len()
<bound method Residue.atoms of <Residue 0 (ALA) of chain 0>>
<bound method Residue.atoms of <Residue 0 (ALA) of chain 0>>
